# 🏆 Team Analysis - IPL (2008-2025)

Comprehensive analysis of team performance, win rates, and strategic insights across IPL seasons.

In [1]:
import pathlib
import sys


def find_project_root(start: pathlib.Path) -> pathlib.Path:
    for candidate in [start, *start.parents]:
        if (candidate / "src").exists() and (candidate / "data").exists():
            return candidate
    return start


ROOT_DIR = find_project_root(pathlib.Path.cwd().resolve())
sys.path.append(str(ROOT_DIR))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

from src.data_loader import load_data
from src.analytics.team import (
    team_summary, team_top_batter, team_top_bowler,
    team_win_by_season, team_win_percentage
)

# Set styling
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (14, 6)

# Load data
matches, deliveries = load_data()
print("✅ Team Data loaded successfully!")
print(f"Matches: {len(matches)}, Deliveries: {len(deliveries)}")


✅ Team Data loaded successfully!
Matches: 1169, Deliveries: 278205


## 1. Overall Team Win Percentage

In [2]:
# Calculate win percentage for each team
all_teams = sorted(set(list(matches['team1'].unique()) + list(matches['team2'].unique())))
win_data = []

for team in all_teams:
    team_matches = matches[(matches['team1'] == team) | (matches['team2'] == team)]
    wins = len(team_matches[team_matches['winner'] == team])
    total = len(team_matches)
    win_pct = (wins / total * 100) if total > 0 else 0
    win_data.append({'Team': team, 'Wins': wins, 'Matches': total, 'Win %': win_pct})

win_df = pd.DataFrame(win_data).sort_values('Win %', ascending=False)

fig = px.bar(
    win_df,
    x='Team',
    y='Win %',
    title='Team Win Percentage (All Time)',
    labels={'Win %': 'Win Percentage (%)'},
    color='Win %',
    color_continuous_scale='RdYlGn',
    text='Win %'
)
fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig.update_layout(height=500, showlegend=False)
fig.show()

print("🏆 Team Win Percentage Rankings:")
print(win_df.to_string())

🏆 Team Win Percentage Rankings:
                           Team  Wins  Matches      Win %
4                Gujarat Titans    37       60  61.666667
0           Chennai Super Kings   142      252  56.349206
8                Mumbai Indians   151      277  54.512635
7          Lucknow Super Giants    30       58  51.724138
6         Kolkata Knight Riders   135      264  51.136364
12  Royal Challengers Bengaluru   132      270  48.888889
10             Rajasthan Royals   114      235  48.510638
13          Sunrisers Hyderabad    93      196  47.448980
9                  Punjab Kings   119      264  45.075758
2                Delhi Capitals   118      267  44.194757
3                 Gujarat Lions    13       30  43.333333
5          Kochi Tuskers Kerala     6       14  42.857143
1               Deccan Chargers    29       75  38.666667
11       Rising Pune Supergiant    27       76  35.526316


## 2. Team Performance by Season

In [3]:
# Team performance progression
season_performance = []

for team in all_teams:
    for season in sorted(matches['season'].unique()):
        team_season = matches[((matches['team1'] == team) | (matches['team2'] == team)) & (matches['season'] == season)]
        if len(team_season) > 0:
            wins = len(team_season[team_season['winner'] == team])
            win_pct = (wins / len(team_season) * 100)
            season_performance.append({'Team': team, 'Season': season, 'Win %': win_pct})

season_perf_df = pd.DataFrame(season_performance)

# Select top 5 teams for visualization
top_teams = win_df.head(5)['Team'].tolist()
top_season_df = season_perf_df[season_perf_df['Team'].isin(top_teams)]

fig = px.line(
    top_season_df,
    x='Season',
    y='Win %',
    color='Team',
    title='Win % Trend - Top 5 Teams',
    markers=True,
    height=500
)
fig.update_layout(hovermode='x unified')
fig.show()

print("📊 Top 5 Teams Season Performance:")
for team in top_teams:
    print(f"\n{team}:")
    print(season_perf_df[season_perf_df['Team'] == team].to_string(index=False))

📊 Top 5 Teams Season Performance:

Gujarat Titans:
          Team Season     Win %
Gujarat Titans   2022 75.000000
Gujarat Titans   2023 64.705882
Gujarat Titans   2024 41.666667
Gujarat Titans   2025 60.000000

Chennai Super Kings:
               Team  Season     Win %
Chennai Super Kings 2007/08 56.250000
Chennai Super Kings    2009 57.142857
Chennai Super Kings 2009/10 56.250000
Chennai Super Kings    2011 68.750000
Chennai Super Kings    2012 55.555556
Chennai Super Kings    2013 66.666667
Chennai Super Kings    2014 62.500000
Chennai Super Kings    2015 58.823529
Chennai Super Kings    2018 68.750000
Chennai Super Kings    2019 58.823529
Chennai Super Kings 2020/21 42.857143
Chennai Super Kings    2021 68.750000
Chennai Super Kings    2022 28.571429
Chennai Super Kings    2023 62.500000
Chennai Super Kings    2024 50.000000
Chennai Super Kings    2025 28.571429

Mumbai Indians:
          Team  Season     Win %
Mumbai Indians 2007/08 50.000000
Mumbai Indians    2009 38.461538
Mumba

## 3. Toss Impact Analysis

In [4]:
# Toss impact analysis
toss_data = []

for team in all_teams:
    team_matches = matches[(matches['team1'] == team) | (matches['team2'] == team)]
    
    # Wins when toss won
    toss_won = team_matches[team_matches['toss_winner'] == team]
    wins_after_toss = len(toss_won[toss_won['winner'] == team])
    toss_won_total = len(toss_won)
    
    # Wins when toss lost
    toss_lost = team_matches[team_matches['toss_winner'] != team]
    wins_after_loss = len(toss_lost[toss_lost['winner'] == team])
    toss_lost_total = len(toss_lost)
    
    toss_data.append({
        'Team': team,
        'Toss Won & Match Won': wins_after_toss,
        'Toss Lost & Match Won': wins_after_loss,
        'Toss Win Rate': (wins_after_toss / toss_won_total * 100) if toss_won_total > 0 else 0,
        'No-Toss Win Rate': (wins_after_loss / toss_lost_total * 100) if toss_lost_total > 0 else 0
    })

toss_df = pd.DataFrame(toss_data)

fig = go.Figure()
fig.add_trace(go.Bar(x=toss_df['Team'], y=toss_df['Toss Win Rate'], name='After Winning Toss', marker_color='#1f77b4'))
fig.add_trace(go.Bar(x=toss_df['Team'], y=toss_df['No-Toss Win Rate'], name='After Losing Toss', marker_color='#ff7f0e'))

fig.update_layout(
    title='Toss Impact on Match Wins',
    xaxis_title='Team',
    yaxis_title='Win % (when toss won/lost)',
    barmode='group',
    height=500
)
fig.show()

print("🎲 Toss Impact Analysis:")
print(toss_df.to_string())

🎲 Toss Impact Analysis:
                           Team  Toss Won & Match Won  Toss Lost & Match Won  Toss Win Rate  No-Toss Win Rate
0           Chennai Super Kings                    78                     64      60.937500         51.612903
1               Deccan Chargers                    19                     10      44.186047         31.250000
2                Delhi Capitals                    64                     54      46.376812         41.860465
3                 Gujarat Lions                    10                      3      66.666667         20.000000
4                Gujarat Titans                    19                     18      65.517241         58.064516
5          Kochi Tuskers Kerala                     4                      2      50.000000         33.333333
6         Kolkata Knight Riders                    71                     64      55.468750         47.058824
7          Lucknow Super Giants                    13                     17      54.166667     

## 4. Head-to-Head Records

In [5]:
# Create head-to-head matrix
teams = sorted(all_teams)
h2h_matrix = pd.DataFrame(0, index=teams, columns=teams)

for _, match in matches.iterrows():
    team1 = match['team1']
    team2 = match['team2']
    winner = match['winner']
    
    if winner == team1:
        h2h_matrix.loc[team1, team2] += 1
    elif winner == team2:
        h2h_matrix.loc[team2, team1] += 1

# Heatmap
fig = px.imshow(
    h2h_matrix,
    labels=dict(color='Wins'),
    title='Head-to-Head: Wins Against Each Team',
    color_continuous_scale='YlOrRd',
    height=700,
    width=900
)
fig.update_layout(
    xaxis_title='Opponent',
    yaxis_title='Team'
)
fig.show()

print("📊 Head-to-Head Matrix (Wins):")
print(h2h_matrix)

📊 Head-to-Head Matrix (Wins):
                             Chennai Super Kings  Deccan Chargers  \
Chennai Super Kings                            0                6   
Deccan Chargers                                4                0   
Delhi Capitals                                12                7   
Gujarat Lions                                  0                0   
Gujarat Titans                                 4                0   
Kochi Tuskers Kerala                           1                0   
Kolkata Knight Riders                         11                7   
Lucknow Super Giants                           3                0   
Mumbai Indians                                21                6   
Punjab Kings                                  15                7   
Rajasthan Royals                              15                7   
Rising Pune Supergiant                         2                1   
Royal Challengers Bengaluru                   13                5   
Sunr

## 5. Top Performers by Team

In [6]:
# Top batsmen and bowlers for top 3 teams
top_3_teams = win_df.head(3)['Team'].tolist()

for team in top_3_teams:
    print(f"\n{'='*60}")
    print(f"🏆 {team} Top Performers")
    print(f"{'='*60}")
    
    top_batsmen = team_top_batter(deliveries, team).head(5)
    print(f"\n📊 Top 5 Run Scorers:")
    print(top_batsmen.to_string())
    
    top_bowlers = team_top_bowler(deliveries, team).head(5)
    print(f"\n🎯 Top 5 Wicket Takers:")
    print(top_bowlers.to_string())


🏆 Gujarat Titans Top Performers

📊 Top 5 Run Scorers:
           batsman  batsman_runs
0     Shubman Gill          2449
1  B Sai Sudharsan          1793
2        DA Miller           950
3        HH Pandya           833
4          WP Saha           824

🎯 Top 5 Wicket Takers:
              bowler  wickets
0        Rashid Khan       67
1     Mohammed Shami       49
2          MM Sharma       47
3      R Sai Kishore       32
4  M Prasidh Krishna       26

🏆 Chennai Super Kings Top Performers

📊 Top 5 Run Scorers:
        batsman  batsman_runs
0      MS Dhoni          4865
1      SK Raina          4695
2  F du Plessis          2721
3    RD Gaikwad          2502
4     RA Jadeja          2198

🎯 Top 5 Wicket Takers:
      bowler  wickets
0   DJ Bravo      158
1  RA Jadeja      149
2   R Ashwin      106
3  JA Morkel       86
4  DL Chahar       78

🏆 Mumbai Indians Top Performers

📊 Top 5 Run Scorers:
        batsman  batsman_runs
0     RG Sharma          5878
1      SA Yadav          3703
2 